# Distilling the Knowledge in a Neural Network

Replication of Hinton, Vinyals and Dean (2015), *Distilling the Knowledge in a Neural
Network*, NeurIPS Deep Learning Workshop.

We first train a large 'teacher' network on the full MNIST training set, then train a small
'student' on a limited transfer set (a 2000-image subset) two ways: directly on the hard
labels, and via knowledge distillation on the teacher's temperature-softened class
probabilities. With limited data the hard labels alone are not very informative, so this is
exactly the regime where the paper's 'dark knowledge' should help: the teacher's soft targets
encode what it learned from the full dataset. We reproduce the result that the distilled
student generalizes better than the one trained on hard labels alone.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision as tv, torchvision.transforms as T
torch.manual_seed(0)

In [2]:
tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
train = tv.datasets.MNIST("../data", train=True,  download=True, transform=tf)
test  = tv.datasets.MNIST("../data", train=False, download=True, transform=tf)
train_dl = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)          # full set: teacher
transfer = torch.utils.data.Subset(train, range(2000))                              # limited set: student
transfer_dl = torch.utils.data.DataLoader(transfer, batch_size=128, shuffle=True)
test_dl  = torch.utils.data.DataLoader(test,  batch_size=512)

def accuracy(net):
    net.eval()
    with torch.no_grad():
        return sum((net(x).argmax(1) == y).sum().item() for x, y in test_dl) / len(test)

In [3]:
# Teacher: wide MLP. Student: much smaller MLP.
def teacher(): return nn.Sequential(nn.Flatten(), nn.Linear(784,1200), nn.ReLU(),
                                     nn.Linear(1200,1200), nn.ReLU(), nn.Linear(1200,10))
def student(): return nn.Sequential(nn.Flatten(), nn.Linear(784,32), nn.ReLU(), nn.Linear(32,10))
print("teacher params:", sum(p.numel() for p in teacher().parameters()))
print("student params:", sum(p.numel() for p in student().parameters()))

teacher params: 2395210
student params: 25450


In [4]:
# Train the teacher on hard labels.
T_net = teacher(); opt = torch.optim.Adam(T_net.parameters(), lr=1e-3); lf = nn.CrossEntropyLoss()
for ep in range(4):
    T_net.train()
    for x, y in train_dl:
        opt.zero_grad(); lf(T_net(x), y).backward(); opt.step()
    print(f"teacher epoch {ep+1}: test_acc={accuracy(T_net):.4f}")
teacher_acc = accuracy(T_net)

teacher epoch 1: test_acc=0.9612


teacher epoch 2: test_acc=0.9721


teacher epoch 3: test_acc=0.9775


teacher epoch 4: test_acc=0.9781


In [5]:
# Baseline student trained only on the hard labels of the limited transfer set.
S1 = student(); opt = torch.optim.Adam(S1.parameters(), lr=1e-3)
for ep in range(60):
    S1.train()
    for x, y in transfer_dl:
        opt.zero_grad(); lf(S1(x), y).backward(); opt.step()
baseline_acc = accuracy(S1)
print(f"student (hard labels only): test_acc={baseline_acc:.4f}")

student (hard labels only): test_acc=0.8950


In [6]:
# Distilled student: blend hard-label loss with KL to the teacher's softened targets.
TEMP, ALPHA = 4.0, 0.7
S2 = student(); opt = torch.optim.Adam(S2.parameters(), lr=1e-3)
T_net.eval()
for ep in range(60):
    S2.train()
    for x, y in transfer_dl:
        with torch.no_grad():
            soft_t = F.softmax(T_net(x) / TEMP, dim=1)         # teacher dark knowledge
        logits = S2(x)
        soft_s = F.log_softmax(logits / TEMP, dim=1)
        kd = F.kl_div(soft_s, soft_t, reduction="batchmean") * (TEMP ** 2)
        loss = ALPHA * kd + (1 - ALPHA) * lf(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
    if (ep+1) % 10 == 0: print(f"distill epoch {ep+1}: test_acc={accuracy(S2):.4f}")
distilled_acc = accuracy(S2)

distill epoch 10: test_acc=0.8748


distill epoch 20: test_acc=0.8904


distill epoch 30: test_acc=0.8981


distill epoch 40: test_acc=0.8984


distill epoch 50: test_acc=0.9010


distill epoch 60: test_acc=0.9031


In [7]:
print(f"teacher (1200-1200) accuracy : {teacher_acc*100:.2f}%")
print(f"student hard-label accuracy  : {baseline_acc*100:.2f}%")
print(f"student distilled accuracy   : {distilled_acc*100:.2f}%")
print(f"distillation gain over hard labels: {(distilled_acc-baseline_acc)*100:+.2f} pp")

teacher (1200-1200) accuracy : 97.81%
student hard-label accuracy  : 89.50%
student distilled accuracy   : 90.31%
distillation gain over hard labels: +0.81 pp
